# Databricks & AzureML Integration - Complete Guide

This guide provides links and overviews of all notebooks demonstrating Databricks and Azure Machine Learning integration patterns.

## 📚 Notebook Index

### Original Notebooks
These notebooks demonstrate basic connectivity between the two services:

1. **[AzureML_to_Databricks_Data_Access.ipynb](AzureML_to_Databricks_Data_Access.ipynb)**
   - AzureML environment calling Databricks model serving endpoints
   - Uses AAD tokens for authentication
   - Prerequisites: Network connectivity, Databricks endpoint

2. **[Databricks_to_AzureML_Connection.ipynb](Databricks_to_AzureML_Connection.ipynb)**
   - Databricks runtime calling AzureML managed online endpoints
   - Managed identity or service principal authentication
   - Real-time model scoring from Databricks

### New Integration Notebooks

#### 3. **[01_Feature_Engineering_Preparation.ipynb](01_Feature_Engineering_Preparation.ipynb)**
**Focus:** Data preparation for ML model training

Key capabilities:
- Load raw data from Azure Storage
- Data cleaning and transformation using Spark
- Feature engineering and derived features
- Feature Store creation in Unity Catalog
- Export data for AzureML (CSV/Parquet formats)
- Data quality validation

**Use case:** When you need to prepare and engineer features in Databricks before model training

---

#### 4. **[02_Model_Training_Pipeline.ipynb](02_Model_Training_Pipeline.ipynb)**
**Focus:** Train models in Databricks and register to AzureML

Key capabilities:
- Load features from Unity Catalog
- Train scikit-learn/MLlib models
- Evaluate model performance (accuracy, precision, recall, F1)
- MLflow model registration and versioning
- Transition models through stages (Development → Production)
- Export model artifacts for AzureML
- Generate model metadata and feature importance

**Use case:** Building end-to-end training pipeline with experiment tracking and model versioning

---

#### 5. **[03_Batch_Prediction_Scoring.ipynb](03_Batch_Prediction_Scoring.ipynb)**
**Focus:** Large-scale batch predictions using AzureML models in Databricks

Key capabilities:
- Load data for scoring from Unity Catalog
- Authenticate to AzureML endpoints
- Distribute scoring across Spark clusters (Pandas UDF)
- Handle batch processing at scale
- Store predictions back to Unity Catalog
- Error handling and retry logic
- Scoring statistics and performance analysis

**Use case:** Production batch scoring on millions of records using distributed computing

---

#### 6. **[04_MLOps_Orchestration.ipynb](04_MLOps_Orchestration.ipynb)**
**Focus:** Automated ML pipeline orchestration and monitoring

Key capabilities:
- Data validation and quality checks
- Feature engineering with metrics logging
- Model training with experiment tracking
- Model registration in AzureML
- Workflow execution logging and reporting
- Webhook notifications for pipeline events
- Scheduled workflow templates

**Use case:** Enterprise ML workflows with automated retraining, monitoring, and alerting

---

#### 7. **[05_Real_Time_Inference.ipynb](05_Real_Time_Inference.ipynb)**
**Focus:** Real-time model serving and inference endpoints

Key capabilities:
- Create AzureML online endpoints
- Single-record real-time inference
- Batch inference via REST API
- Latency and performance monitoring
- Local model serving for testing
- Error handling and retry policies
- Key-based authentication

**Use case:** Production-grade real-time API for applications requiring instant predictions

---

#### 8. **[06_Unity_Catalog_Integration.ipynb](06_Unity_Catalog_Integration.ipynb)**
**Focus:** Data governance and secure sharing via Unity Catalog

Key capabilities:
- Create unified metadata with Unity Catalog
- Medallion architecture (Bronze/Silver/Gold layers)
- Metadata tagging and lineage tracking
- PII classification and protection
- External data sharing (to AzureML)
- Access control and permissions
- Data quality monitoring
- Secret management for credentials

**Use case:** Enterprise data governance, data sharing, and compliance across Databricks and AzureML

---

## 🔄 Common Workflows

### Workflow 1: Train in Databricks, Serve in AzureML
```
01_Feature_Engineering → 02_Model_Training → 05_Real_Time_Inference
```
- Prepare features in Databricks
- Train and register model
- Deploy and serve via AzureML endpoint

### Workflow 2: Batch Scoring Pipeline
```
01_Feature_Engineering → 03_Batch_Prediction_Scoring → Unity Catalog
```
- Engineer features for all records
- Score at scale using AzureML model
- Store results for reporting/analytics

### Workflow 3: Automated MLOps Pipeline
```
04_MLOps_Orchestration (includes all steps)
```
- Automated data validation
- Feature engineering
- Model training and registration
- Scheduled retraining
- Monitoring and alerts

### Workflow 4: Secure Data Sharing
```
01_Feature_Engineering → 06_Unity_Catalog_Integration → External Access
```
- Prepare features with governance
- Organize in Unity Catalog
- Share securely with AzureML
- Track lineage and access

## ⚙️ Setup & Prerequisites

### Required Environment Variables
```bash
# Azure Authentication
AZURE_SUBSCRIPTION_ID=<subscription-id>
AZURE_RESOURCE_GROUP=<resource-group>
AZURE_TENANT_ID=<tenant-id>
AZURE_CLIENT_ID=<service-principal-id>
AZURE_CLIENT_SECRET=<service-principal-secret>

# AzureML Configuration
AZUREML_WORKSPACE_NAME=<workspace-name>
AZUREML_ENDPOINT_URL=<endpoint-scoring-uri>

# Databricks Configuration
DATABRICKS_HOST=https://adb-<id>.<region>.azuredatabricks.net
DATABRICKS_ENDPOINT_NAME=<model-endpoint-name>

# Storage
STORAGE_ACCOUNT_NAME=<storage-account>
e STORAGE_CONTAINER_NAME=<container-name>
STORAGE_ACCOUNT_KEY=<storage-key>

# Optional: Notifications
MLOPS_WEBHOOK_URL=<webhook-url>
```

### AzureML Datastore for Storage Account
Use a datastore to access your data in Azure Storage from AzureML jobs. Prefer managed identity or SAS over account keys if available.

```python
import os
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient
from azure.ai.ml.entities import AzureBlobDatastore, AccountKeyConfiguration

subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
workspace_name = os.getenv("AZUREML_WORKSPACE_NAME")
account_name = os.getenv("STORAGE_ACCOUNT_NAME")
container_name = os.getenv("STORAGE_CONTAINER_NAME")
account_key = os.getenv("STORAGE_ACCOUNT_KEY")

if not all([subscription_id, resource_group, workspace_name, account_name, container_name, account_key]):
    raise ValueError("Missing AzureML or storage environment variables.")

ml_client = MLClient(DefaultAzureCredential(exclude_interactive_browser_credential=True), subscription_id, resource_group, workspace_name)
datastore = AzureBlobDatastore(
    name="ml-data",
    account_name=account_name,
    container_name=container_name,
    credentials=AccountKeyConfiguration(account_key=account_key),
)
ml_client.datastores.create_or_update(datastore)
print(f"Datastore created: {datastore.name}")
```

### Required Python Packages
```bash
pip install azure-ai-ml azure-identity requests mlflow pandas scikit-learn pyspark
```

### Databricks Cluster Configuration
- Runtime: Databricks Runtime ML 13.0+
- Python: 3.10+
- Libraries: MLflow, scikit-learn, azure-identity

## 📊 Architecture Diagram

```
┌─────────────────────────────────────────────────────────────┐
│                     DATABRICKS WORKSPACE                       │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────────┐  │
│  │  Notebook│  │  MLflow  │  │  Spark   │  │Unity Catalog │  │
│  │  Cluster │━━┃ Tracking ┃━━┃ Cluster  ┃━━┃  (Metadata)  ┃  │
│  └──────────┘  └──────────┘  └──────────┘  └──────────────┘  │
│       │              │              │              │           │
│       └──────────────┼──────────────┼──────────────┘           │
│                      │              │                          │
└──────────────────────┼──────────────┼──────────────────────────┘
                       │              │
                 ┌─────▼──────┐ ┌────▼─────┐
                 │   Models   │ │   Data   │
                 │   Export   │ │  Export  │
                 └─────┬──────┘ └────┬─────┘
                       │              │
    ┌──────────────────▼──────────────▼──────────────────┐
    │   Azure Storage (ADLS Gen2)                        │
    │  ┌─────────────────────────────────────────────┐  │
    │  │  Models/ (model artifacts)                  │  │
    │  │  Data/   (features, training sets)          │  │
    │  └─────────────────────────────────────────────┘  │
    └──────────────────┬───────────────────────────────┘
                       │
                       ▼
    ┌─────────────────────────────────────┐
    │      AZURE ML SERVICE               │
    │  ┌─────────────────────────────┐    │
    │  │  Model Registry             │    │
    │  │  Online Endpoints           │    │
    │  │  Batch Endpoints            │    │
    │  │  Monitoring & Logging       │    │
    │  └─────────────────────────────┘    │
    └─────────────────┬────────────────────┘
                      │
            ┌─────────▼──────────┐
            │  REST API Endpoints │
            │  (Real-time/Batch)  │
            └─────────────────────┘
```

## 🚀 Quick Start

### Option 1: Simple Integration (5 minutes)
1. Run `AzureML_to_Databricks_Data_Access.ipynb` - Test connectivity
2. Run `Databricks_to_AzureML_Connection.ipynb` - Test reverse connectivity

### Option 2: Training Pipeline (30 minutes)
1. Run `01_Feature_Engineering_Preparation.ipynb` - Prepare data
2. Run `02_Model_Training_Pipeline.ipynb` - Train model
3. Run `05_Real_Time_Inference.ipynb` - Deploy endpoint

### Option 3: Full MLOps (1 hour)
1. Run `04_MLOps_Orchestration.ipynb` - Complete pipeline
2. Run `06_Unity_Catalog_Integration.ipynb` - Add governance
3. Run `03_Batch_Prediction_Scoring.ipynb` - Batch scoring

### Option 4: Enterprise Setup (2 hours)
1. Complete Option 3
2. Configure access controls in Unity Catalog
3. Set up monitoring dashboards
4. Configure automated retraining schedules

## 📝 Troubleshooting

### Common Issues

**Issue:** Authentication failure
- Check environment variables are set
- Verify service principal has correct permissions
- Use `DefaultAzureCredential` for Managed Identity

**Issue:** Network connectivity problems
- Ensure VNet/Private Links are configured
- Check security groups allow traffic
- Verify endpoint URLs are correct

**Issue:** Model not found in registry
- Verify MLflow tracking server configuration
- Check model was registered successfully
- Confirm model stage transitions (Development → Production)

**Issue:** Slow batch scoring
- Increase batch size (larger requests, fewer API calls)
- Use Pandas UDF for distributed processing
- Consider batch endpoint instead of online endpoint

**Issue:** Data not visible in Unity Catalog
- Check CATALOG and SCHEMA names match configuration
- Verify permissions are granted
- Ensure workspace has Unity Catalog enabled